# Qualidade das noites pelo fluxo bruto

Compara noites com as métricas próprias de `night_metrics` (validadas em `02_respiratory_validation.ipynb`), sem depender do AHI da ResMed.

| Métrica | O que mede | Melhor |
|---|---|---|
| `surge_index` | microdespertares estimados por hora | menor |
| `hypopnea30_index` | apneias + hipopneias com queda de 30% por hora | menor |
| `rera_like_index` | surtos precedidos de respirações achatadas, por hora | menor |
| `flow_limited_pct` | % de respirações com limitação de fluxo | menor |
| `stable_pct` | % das janelas de 5 min com respiração estável e sem eventos | maior |

Só entram na comparação noites com a mesma pressão (7,0 cmH2O) e pelo menos 4 h de sono estimado.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sleep_analysis.db import connect
from sleep_analysis.study import label_period, load_study

PRESSURE = 7.0
MIN_SLEEP_HOURS = 4

db = connect()
study = load_study()

metrics = db.sql("""
    SELECT m.*, n.pressure, n.ahi, n.leak_95_l_min, n.usage_hours
    FROM night_metrics m
    JOIN nights n USING (night)
    ORDER BY night
""").df()
metrics["period"] = label_period(metrics, study)
metrics["event_index"] = metrics["hypopnea30_index"] + metrics["rera_like_index"]
comparable = metrics[(metrics["pressure"] == PRESSURE) & (metrics["sleep_hours"] >= MIN_SLEEP_HOURS)].copy()
print(f"{len(metrics)} noites com fluxo; {len(comparable)} comparáveis a {PRESSURE} cmH2O")
comparable.groupby("period").size()

## Ranking das noites

Índice composto = média dos z-scores de `surge_index`, `event_index` (`hypopnea30_index` + `rera_like_index`) e `stable_pct` com sinal invertido. Os z-scores são calculados contra as noites comparáveis de antes da cirurgia. **Menor é melhor**; 0 é uma noite típica.

In [ ]:
# +1: maior é pior; -1: maior é melhor
COMPOSITE = {"surge_index": 1, "event_index": 1, "stable_pct": -1}
reference = comparable[comparable["period"] == "before"]
for column, direction in COMPOSITE.items():
    comparable[f"z_{column}"] = direction * (comparable[column] - reference[column].mean()) / reference[column].std()
comparable["composite"] = comparable[[f"z_{column}" for column in COMPOSITE]].mean(axis=1)

ranking_columns = [
    "night", "composite", "surge_index", "hypopnea30_index", "rera_like_index",
    "flow_limited_pct", "stable_pct", "sleep_hours", "ahi",
]
ranked = comparable.sort_values("composite")[ranking_columns].set_index("night").round(2)
print("Melhores noites")
display(ranked.head(10))
print("Piores noites")
display(ranked.tail(10))

## Tendência ao longo do tempo

Pontos cinza: cada noite (todas as pressões); linha azul: mediana móvel de 7 dias das noites comparáveis (com ao menos 3 noites na janela). A linha vermelha marca a cirurgia, quando definida em `config/study.json`.

In [ ]:
panels = {
    "surge_index": "Surtos/h",
    "hypopnea30_index": "Hipopneias 30%/h",
    "rera_like_index": "RERA-like/h",
    "stable_pct": "Janelas estáveis (%)",
    "ahi": "AHI ResMed",
}
fig, axes = plt.subplots(len(panels), 1, figsize=(12, 12), sharex=True)
for ax, (column, title) in zip(axes, panels.items()):
    ax.plot(metrics["night"], metrics[column], ".", color="gray", alpha=0.4)
    # Reindexa por dia para a linha quebrar nos períodos sem noites comparáveis
    daily = comparable.set_index("night")[column].asfreq("D")
    ax.plot(daily.index, daily.rolling(7, min_periods=3).median(), color="tab:blue")
    ax.set_ylabel(title)
    if study.surgery_date is not None:
        ax.axvline(study.surgery_date, color="tab:red", linestyle="--")
axes[0].set_title("Métricas do fluxo bruto por noite")
fig.tight_layout()

## Terços da noite

O 1º terço concentra o sono profundo e o último, o REM. Comparar as melhores e as piores noites por terço mostra **onde** a noite ruim perde qualidade.

In [ ]:
third_columns = [
    f"{metric}_t{third}" for metric in ["stable_pct", "surge_index", "flow_limited_pct"] for third in (1, 2, 3)
]
quartile = pd.qcut(comparable["composite"], 4, labels=["melhores 25%", "2º quartil", "3º quartil", "piores 25%"])
comparable.groupby(quartile, observed=True)[third_columns].median().T.round(2)

## Relação com o AHI e com o vazamento

Correlação de Spearman nas noites comparáveis. Correlação baixa com o AHI indica que as métricas próprias captam algo que o AHI não mostra. Correlação alta com o vazamento indicaria que a métrica está contaminada pela máscara.

In [ ]:
correlation_columns = [
    "composite", "surge_index", "hypopnea30_index", "rera_like_index", "flow_limited_pct", "stable_pct",
    "wake_like_bouts", "ahi", "leak_95_l_min", "sleep_hours",
]
comparable[correlation_columns].corr(method="spearman").round(2)

## Antes e depois da cirurgia

Linha de base: as últimas `baselineNights` noites comparáveis antes da cirurgia. Depois: noites comparáveis após os `recoveryDaysExcluded` dias de recuperação. A diferença é de medianas (depois − antes), com intervalo de confiança de 95% por bootstrap. Se o intervalo não cruza zero, a mudança é consistente.

Enquanto a data da cirurgia não estiver em `config/study.json`, a célula mostra só a linha de base atual.

In [ ]:
COMPARE = ["composite", "surge_index", "hypopnea30_index", "rera_like_index", "flow_limited_pct", "stable_pct", "ahi"]
BOOTSTRAP_SAMPLES = 5000
rng = np.random.default_rng(0)


def bootstrap_median_difference(before: np.ndarray, after: np.ndarray) -> tuple[float, float]:
    differences = [
        np.median(rng.choice(after, after.size)) - np.median(rng.choice(before, before.size))
        for _ in range(BOOTSTRAP_SAMPLES)
    ]
    return tuple(np.percentile(differences, [2.5, 97.5]))


baseline = comparable[comparable["period"] == "before"].tail(study.baseline_nights)
after = comparable[comparable["period"] == "after"]
print(f"Linha de base: {len(baseline)} noites ({baseline['night'].min():%d/%m} a {baseline['night'].max():%d/%m})")

if after.empty:
    display(baseline[COMPARE].describe().loc[["count", "25%", "50%", "75%"]].round(2))
else:
    rows = []
    for column in COMPARE:
        low, high = bootstrap_median_difference(baseline[column].dropna().to_numpy(), after[column].dropna().to_numpy())
        rows.append(
            {
                "métrica": column,
                "antes (mediana)": baseline[column].median(),
                "depois (mediana)": after[column].median(),
                "diferença": after[column].median() - baseline[column].median(),
                "IC 95% baixo": low,
                "IC 95% alto": high,
            }
        )
    print(f"Depois: {len(after)} noites")
    display(pd.DataFrame(rows).round(2))